In [ ]:
import os
import glob

def read_asc_content(asc_path):
    """Read .asc file into memory with automatic encoding detection."""
    encodings = ['utf-16', 'utf-16-le', 'utf-16-be', 'latin-1', 'ascii', 'utf-8']
    
    for encoding in encodings:
        try:
            with open(asc_path, 'r', encoding=encoding) as f:
                content = f.read()
                # Verify it's valid LTspice content
                if 'Version' in content or 'SHEET' in content or 'WIRE' in content:
                    return content
        except (UnicodeDecodeError, UnicodeError):
            continue
    
    raise ValueError(f"Could not decode {asc_path} with any known encoding")

def parse_asc_content(content):
    """Extract components from the raw text content of the .asc file."""
    components = []
    current_component = None
    lines = content.splitlines()

    for line in lines:
        line_clean = line.strip()
        line_upper = line_clean.upper()
        
        if line_upper.startswith('SYMBOL'):
            if current_component is not None:
                components.append(current_component)
            
            parts = line_clean.split()
            if len(parts) >= 2:
                current_component = {'Symbol_Type': parts[1]}
            else:
                current_component = {'Symbol_Type': 'Unknown'}
                
        elif line_upper.startswith('SYMATTR '):
            if current_component is not None:
                parts = line_clean.split(maxsplit=2)
                if len(parts) >= 3:
                    attr_name = parts[1]
                    attr_value = parts[2]
                    current_component[attr_name] = attr_value

    if current_component is not None:
        components.append(current_component)
            
    return components

def get_cost(sym_type):
    """Calculate cost based on component type."""
    sym_type = str(sym_type).lower()
    if sym_type in ['npn', 'pnp']: 
        return 5.0
    elif sym_type in ['pmos', 'nmos']:
        return 3.0
    elif sym_type == 'res': 
        return 10.0
    elif sym_type == 'cap': 
        return 40.0
    else:
        return 0.0

# --- Execution Block ---
TARGET_FOLDER = "." 
search_pattern = os.path.join(TARGET_FOLDER, '*.asc')
asc_files = glob.glob(search_pattern)

for file in asc_files:
    filename = os.path.basename(file)
    try:
        # Read content handling encoding, but keeping it entirely in memory
        content = read_asc_content(file)
        components = parse_asc_content(content)
        
        # Calculate total cost
        total_cost = sum(get_cost(comp.get('Symbol_Type', '')) for comp in components)-10
        
        # Print only the filename and the cost
        print(f"{filename}: {total_cost}")
        
    except Exception as e:
        # Fails silently for non-parseable files to stick to the "only print cost" rule
        pass
